# BLIP MSVD Retrieval: Zero-Shot vs Fine-Tuned

This streamlined notebook runs the complete MSVD experiment and reports only the final comparison between zero-shot BLIP and fine-tuned BLIP.

Removed from this version: caption generation, BLEU/CIDEr, runtime evaluation, confusion metrics, and visualization cells.

The training loss is video-aware: captions and sampled frames belonging to the same video are treated as multiple valid positives rather than false negatives.

In [ ]:
!pip install -q transformers accelerate decord pillow tqdm pandas

In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForImageTextRetrieval
from decord import VideoReader, cpu

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    print('Not running in Colab; using local paths.')

Mounted at /content/drive


In [ ]:
# Full-data configuration
SEED = 298
DATA_ROOT = '/content/drive/Shared drives/DATA 298A/DATA/MSVD'
OUTPUT_DIR = '/content/drive/MyDrive/blip_finetuned_comparison_outputs'
MODEL_NAME = 'Salesforce/blip-itm-base-coco'

FRAME_COUNT = 3
FRAME_SAMPLING = 'begin/mid/end'
IMAGE_SIZE = 224
FRAME_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'frames_224_jpeg')
TOP_K_FRAMES = 2
MAX_TEXT_LEN = 64

SMOKE_TEST = False
RUN_FULL_DATA = True
MAX_VIDEOS = 8              # used only if RUN_FULL_DATA=False
MAX_TRAIN_ITEMS = 600       # used only if RUN_FULL_DATA=False
MAX_VAL_ITEMS = 200         # used only if RUN_FULL_DATA=False

# Recommended starting configuration for a G4/A100-class GPU.
EPOCHS = 5
TRAIN_BATCH_SIZE = 32
EVAL_TEXT_BATCH_SIZE = 64
LEARNING_RATE = 1e-5
ENCODER_LEARNING_RATE = 5e-7
UNFREEZE_LAST_N = 1
EARLY_STOPPING_PATIENCE = 2
TEMPERATURE = 0.07
WEIGHT_DECAY = 0.01
NUM_WORKERS = 0

os.makedirs(OUTPUT_DIR, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('SMOKE_TEST:', SMOKE_TEST)
print('RUN_FULL_DATA:', RUN_FULL_DATA)
print('Output:', OUTPUT_DIR)

Device: cuda
SMOKE_TEST: False
RUN_FULL_DATA: True
Output: /content/drive/MyDrive/blip_finetuned_comparison_outputs


## 1. Load MSVD and extract three standardized frames

The loader uses the video-disjoint `msvd_train.json`, `msvd_val.json`, and `msvd_test.json` splits. Each video is represented by beginning, middle, and ending JPEG frames.

In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def clean_caption(text):
    return ' '.join(str(text).strip().split())

def make_video_index(video_root):
    index = {}
    for path in Path(video_root).iterdir():
        if path.is_file():
            index[path.name] = str(path)
            index[path.stem] = str(path)
    return index

def load_shared_split(split, max_videos=None):
    annotation_path = os.path.join(DATA_ROOT, f'msvd_{split}.json')
    video_root = os.path.join(DATA_ROOT, 'raw_videos')
    if not os.path.exists(annotation_path):
        raise FileNotFoundError(annotation_path)
    if not os.path.isdir(video_root):
        raise FileNotFoundError(video_root)

    video_index = make_video_index(video_root)
    by_video = {}
    for item in load_json(annotation_path):
        video_name = str(item.get('video', item.get('video_name', '')))
        video_path = video_index.get(video_name) or video_index.get(Path(video_name).stem)
        if not video_path:
            continue
        video_id = str(item.get('video_id', Path(video_name).stem))
        raw = item.get('caption', item.get('captions', []))
        captions = [raw] if isinstance(raw, str) else raw
        captions = [clean_caption(x) for x in captions if clean_caption(x)]
        if video_id not in by_video:
            by_video[video_id] = {
                'video_id': video_id,
                'video_name': video_name,
                'video_path': video_path,
                'captions': []
            }
        by_video[video_id]['captions'].extend(captions)

    videos = list(by_video.values())
    for video in videos:
        video['captions'] = list(dict.fromkeys(video['captions']))
    videos.sort(key=lambda x: x['video_id'])
    return videos[:max_videos] if max_videos else videos

def sample_indices(total_frames):
    if total_frames <= 0:
        raise ValueError('Video has no frames')
    return [0, total_frames // 2, total_frames - 1]

def extract_frames_as_jpegs(video, output_root=FRAME_OUTPUT_DIR):
    names = ['begin', 'mid', 'end']
    video_dir = Path(output_root) / str(video['video_id'])
    video_dir.mkdir(parents=True, exist_ok=True)
    expected = [video_dir / f'{name}.jpg' for name in names]
    if all(path.exists() for path in expected):
        return [str(path) for path in expected]

    reader = VideoReader(video['video_path'], ctx=cpu(0))
    frames = reader.get_batch(sample_indices(len(reader))).asnumpy()
    paths = []
    for name, array in zip(names, frames):
        image = Image.fromarray(array).convert('RGB')
        image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
        path = video_dir / f'{name}.jpg'
        image.save(path, format='JPEG', quality=95)
        paths.append(str(path))
    return paths

def read_sampled_frames(video, frame_paths=None):
    if frame_paths is not None:
        return [Image.open(path).convert('RGB') for path in frame_paths]
    reader = VideoReader(video['video_path'], ctx=cpu(0))
    return [Image.fromarray(x).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
            for x in reader.get_batch(sample_indices(len(reader))).asnumpy()]

limit = None if RUN_FULL_DATA else MAX_VIDEOS
train_videos = load_shared_split('train', limit)
val_videos = load_shared_split('val', limit)
test_videos = load_shared_split('test', limit)

for video in tqdm(train_videos + val_videos + test_videos, desc='Caching frames'):
    video['frame_paths'] = extract_frames_as_jpegs(video)

print({'train': len(train_videos), 'val': len(val_videos), 'test': len(test_videos)})
assert set(x['video_id'] for x in train_videos).isdisjoint(x['video_id'] for x in val_videos)
assert set(x['video_id'] for x in train_videos).isdisjoint(x['video_id'] for x in test_videos)
assert set(x['video_id'] for x in val_videos).isdisjoint(x['video_id'] for x in test_videos)

Caching frames:   0%|          | 0/1970 [00:00<?, ?it/s]

{'train': 1200, 'val': 100, 'test': 670}


## 2. Dataset with video identifiers

The `video_id` is retained so the loss can recognize multiple valid image-caption pairs from the same video.

In [ ]:
processor = BlipProcessor.from_pretrained(MODEL_NAME)

class FrameCaptionDataset(Dataset):
    def __init__(self, videos, processor, max_items=None):
        self.processor = processor
        self.items = []
        for video in videos:
            for caption in video['captions']:
                for frame_index in range(FRAME_COUNT):
                    self.items.append((video, caption, frame_index))
        if max_items is not None:
            self.items = self.items[:max_items]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        video, caption, frame_index = self.items[index]
        frame = read_sampled_frames(video['video_path'], video['frame_paths'])[frame_index]
        encoded = self.processor(
            images=frame,
            text=caption,
            padding='max_length',
            truncation=True,
            max_length=MAX_TEXT_LEN,
            return_tensors='pt'
        )
        output = {key: value.squeeze(0) for key, value in encoded.items()}
        output['video_id'] = video['video_id']
        return output

train_dataset = FrameCaptionDataset(
    train_videos, processor,
    max_items=None if RUN_FULL_DATA else MAX_TRAIN_ITEMS
)
val_dataset = FrameCaptionDataset(
    val_videos, processor,
    max_items=None if RUN_FULL_DATA else MAX_VAL_ITEMS
)

train_loader = DataLoader(
    train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

print('Training examples:', len(train_dataset))
print('Validation examples:', len(val_dataset))

preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Training examples: 126918
Validation examples: 10929


## 3. Video-aware BLIP fine-tuning

The multi-positive loss averages the log probability over every matching video in the batch. This prevents captions and frames from the same video being incorrectly penalized as negatives.

In [ ]:
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(DEVICE)

for parameter in model.vision_model.parameters():
    parameter.requires_grad = False
for parameter in model.text_encoder.parameters():
    parameter.requires_grad = False

vision_blocks = list(getattr(model.vision_model.encoder, 'layers', []))
text_blocks = list(getattr(model.text_encoder.encoder, 'layer', []))
for block in vision_blocks[-UNFREEZE_LAST_N:]:
    for parameter in block.parameters():
        parameter.requires_grad = True
for block in text_blocks[-UNFREEZE_LAST_N:]:
    for parameter in block.parameters():
        parameter.requires_grad = True

projection_parameters = list(model.vision_proj.parameters()) + list(model.text_proj.parameters())
projection_ids = {id(parameter) for parameter in projection_parameters}
encoder_parameters = [
    parameter for parameter in model.parameters()
    if parameter.requires_grad and id(parameter) not in projection_ids
]
trainable_parameters = projection_parameters + encoder_parameters

optimizer = torch.optim.AdamW(
    [
        {'params': projection_parameters, 'lr': LEARNING_RATE},
        {'params': encoder_parameters, 'lr': ENCODER_LEARNING_RATE}
    ],
    weight_decay=WEIGHT_DECAY
)

print('Projection parameters:', sum(p.numel() for p in projection_parameters))
print('Unfrozen encoder parameters:', sum(p.numel() for p in encoder_parameters))

def move_batch(batch):
    return {
        key: value.to(DEVICE, non_blocking=True)
        for key, value in batch.items()
        if key in {'pixel_values', 'input_ids', 'attention_mask'}
    }

def multi_positive_cross_entropy(logits, positive_mask):
    log_probs = F.log_softmax(logits, dim=1)
    positive_mask = positive_mask.float()
    return -((log_probs * positive_mask).sum(dim=1) /
             positive_mask.sum(dim=1).clamp_min(1.0)).mean()

def contrastive_loss(batch, video_ids):
    image_out = model.vision_model(pixel_values=batch['pixel_values'], return_dict=True)
    image_features = F.normalize(
        model.vision_proj(image_out.last_hidden_state[:, 0, :]), dim=-1
    )
    text_out = model.text_encoder(
        input_ids=batch['input_ids'],
        attention_mask=batch['attention_mask'],
        return_dict=True
    )
    text_features = F.normalize(
        model.text_proj(text_out.last_hidden_state[:, 0, :]), dim=-1
    )

    logits = (image_features @ text_features.T) / TEMPERATURE
    positive = torch.tensor(
        [[left == right for right in video_ids] for left in video_ids],
        device=DEVICE, dtype=torch.bool
    )
    return 0.5 * (
        multi_positive_cross_entropy(logits, positive) +
        multi_positive_cross_entropy(logits.T, positive.T)
    )

def run_epoch(loader, training):
    model.train(training)
    total, count = 0.0, 0
    for batch in tqdm(loader, desc='train' if training else 'validation', leave=False):
        video_ids = list(batch['video_id'])
        numeric_batch = move_batch(batch)
        with torch.set_grad_enabled(training):
            loss = contrastive_loss(numeric_batch, video_ids)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(trainable_parameters, 1.0)
                optimizer.step()
        size = numeric_batch['input_ids'].size(0)
        total += loss.item() * size
        count += size
    return total / max(count, 1)

history = []
best_val = float('inf')
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, True)
    val_loss = run_epoch(val_loader, False)
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss}
    history.append(row)
    print(row)
    with open(os.path.join(OUTPUT_DIR, 'training_history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    if val_loss < best_val:
        best_val = val_loss
        best_epoch = epoch
        model.save_pretrained(os.path.join(OUTPUT_DIR, 'best_checkpoint'))
        processor.save_pretrained(os.path.join(OUTPUT_DIR, 'best_checkpoint'))

    if epoch - best_epoch >= EARLY_STOPPING_PATIENCE:
        print('Early stopping: validation loss did not improve.')
        break

print('Best epoch:', best_epoch)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  895MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  895MB            

model.safetensors: downloading bytes:           |  0.00B            

...

validation:   0%|          | 0/342 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 0.559194963498132, 'val_loss': 3.64971334734189}


train:   0%|          | 0/3967 [00:00<?, ?it/s]

validation:   0%|          | 0/342 [00:00<?, ?it/s]

{'epoch': 3, 'train_loss': 0.4892420012211341, 'val_loss': 3.6491535863301294}
Early stopping: validation loss did not improve.
Best epoch: 1


## 4. BLIP retrieval evaluation and comparison

Every MSVD caption is used as a text query. A video is represented by its frame embeddings, and its score is the mean of the two strongest frame similarities. The reported metrics compare the original zero-shot BLIP model with the fine-tuned checkpoint.

In [ ]:
@torch.inference_mode()
def encode_image_frames(model, images):
    inputs = processor(images=images, return_tensors='pt')
    outputs = model.vision_model(
        pixel_values=inputs['pixel_values'].to(DEVICE), return_dict=True
    )
    features = model.vision_proj(outputs.last_hidden_state[:, 0, :])
    return F.normalize(features, dim=-1).cpu()

@torch.inference_mode()
def encode_text_queries(model, texts):
    inputs = processor(
        text=texts, padding=True, truncation=True,
        max_length=MAX_TEXT_LEN, return_tensors='pt'
    )
    outputs = model.text_encoder(
        input_ids=inputs['input_ids'].to(DEVICE),
        attention_mask=inputs['attention_mask'].to(DEVICE),
        return_dict=True
    )
    features = model.text_proj(outputs.last_hidden_state[:, 0, :])
    return F.normalize(features, dim=-1).cpu()

@torch.inference_mode()
def evaluate_retrieval(eval_model, videos):
    eval_model.eval()
    video_ids = []
    video_embeddings = []
    for video in tqdm(videos, desc='Encoding videos'):
        frame_embeddings = encode_image_frames(
            eval_model,
            read_sampled_frames(video['video_path'], video['frame_paths'])
        )
        video_ids.append(video['video_id'])
        video_embeddings.append(frame_embeddings)

    queries = []
    targets = []
    for video in videos:
        for caption in video['captions']:
            queries.append(caption)
            targets.append(video['video_id'])

    query_embeddings = []
    for start in tqdm(range(0, len(queries), EVAL_TEXT_BATCH_SIZE), desc='Encoding queries'):
        query_embeddings.append(encode_text_queries(
            eval_model, queries[start:start + EVAL_TEXT_BATCH_SIZE]
        ))
    query_embeddings = torch.cat(query_embeddings, dim=0)

    ranks = []
    for query_embedding, target in tqdm(zip(query_embeddings, targets), total=len(targets), desc='Ranking'):
        scores = []
        for frame_embeddings in video_embeddings:
            similarities = frame_embeddings @ query_embedding
            k = min(TOP_K_FRAMES, similarities.numel())
            scores.append(torch.topk(similarities, k=k).values.mean().item())
        order = np.argsort(-np.asarray(scores))
        rank = int(np.where(np.asarray(video_ids)[order] == target)[0][0]) + 1
        ranks.append(rank)

    ranks = np.asarray(ranks)
    return {
        'R@1': float(np.mean(ranks <= 1)),
        'R@5': float(np.mean(ranks <= 5)),
        'R@10': float(np.mean(ranks <= 10)),
        'MRR': float(np.mean(1.0 / ranks)),
        'MeanRank': float(np.mean(ranks)),
        'MedianRank': float(np.median(ranks)),
        'num_queries': int(len(ranks)),
        'num_videos': int(len(video_ids)),
        'ranks': ranks.tolist()
    }

fine_tuned_model = BlipForImageTextRetrieval.from_pretrained(
    os.path.join(OUTPUT_DIR, 'best_checkpoint')
).to(DEVICE)
fine_tuned_processor = BlipProcessor.from_pretrained(
    os.path.join(OUTPUT_DIR, 'best_checkpoint')
)

# Use the selected fine-tuned processor during evaluation.
processor = fine_tuned_processor
fine_tuned_metrics = evaluate_retrieval(fine_tuned_model, test_videos)

del fine_tuned_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

zero_shot_model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(DEVICE)
processor = BlipProcessor.from_pretrained(MODEL_NAME)
zero_shot_metrics = evaluate_retrieval(zero_shot_model, test_videos)

metric_names = ['R@1', 'R@5', 'R@10', 'MRR', 'MeanRank', 'MedianRank']
comparison = pd.DataFrame({
    'Metric': metric_names,
    'Zero-shot BLIP': [zero_shot_metrics[name] for name in metric_names],
    'Fine-tuned BLIP': [fine_tuned_metrics[name] for name in metric_names]
})
comparison['Change'] = comparison['Fine-tuned BLIP'] - comparison['Zero-shot BLIP']

display(comparison)

comparison.to_csv(os.path.join(OUTPUT_DIR, 'zero_shot_vs_fine_tuned_comparison.csv'), index=False)
with open(os.path.join(OUTPUT_DIR, 'zero_shot_vs_fine_tuned_comparison.json'), 'w') as f:
    json.dump({
        'zero_shot': zero_shot_metrics,
        'fine_tuned': fine_tuned_metrics,
        'configuration': {
            'dataset': 'MSVD',
            'model': MODEL_NAME,
            'full_data': RUN_FULL_DATA,
            'epochs': EPOCHS,
            'batch_size': TRAIN_BATCH_SIZE,
            'frame_count': FRAME_COUNT,
            'top_k_frames': TOP_K_FRAMES,
            'learning_rate': LEARNING_RATE,
            'encoder_learning_rate': ENCODER_LEARNING_RATE,
            'unfreeze_last_n': UNFREEZE_LAST_N,
            'temperature': TEMPERATURE,
            'multi_positive_loss': True
        }
    }, f, indent=2)

print('Saved comparison files to:', OUTPUT_DIR)

Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

Encoding videos:   0%|          | 0/670 [00:00<?, ?it/s]

Encoding queries:   0%|          | 0/376 [00:00<?, ?it/s]

Ranking:   0%|          | 0/24033 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

Encoding videos:   0%|          | 0/670 [00:00<?, ?it/s]

Encoding queries:   0%|          | 0/376 [00:00<?, ?it/s]

Ranking:   0%|          | 0/24033 [00:00<?, ?it/s]

,Metric,Zero-shot BLIP,Fine-tuned BLIP,Change
0,R@1,0.375692,0.401323,0.025631
1,R@5,0.654933,0.688470,0.033537
2,R@10,0.741189,0.779512,0.038322
3,MRR,0.501605,0.531728,0.030123
4,MeanRank,20.915200,15.916698,-4.998502
5,MedianRank,2.000000,2.000000,0.000000


Saved comparison files to: /content/drive/MyDrive/blip_finetuned_comparison_outputs
